In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * DEEP DIVE — RAW DATA PAIN: Why Flat Tables Are a Database Anti-Pattern
 * ─────────────────────────────────────────────────────────────────────
*/

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * UNDERSTANDING THE PROBLEMS OF DENORMALIZED, SPREADSHEET-STYLE TABLES
 * ─────────────────────────────────────────────────────────────────────
 *
 * TABLE OF CONTENTS:
 * 1. THE COMFORT OF FLAT TABLES ......... Spreadsheet thinking in databases
 * 2. THE SAMPLE TABLE .................. raw_learning_events
 * 3. THE REAL PROBLEMS ................. Inconsistency, duplication, confusion
 * 4. STUDENT DETAILS REPEATED .......... Name inconsistency, update nightmare
 * 5. COURSE DETAILS REPEATED ........... Title changes affect thousands of rows
 * 6. DUPLICATE ENROLLMENT CONFUSION .... No business rule enforcement
 * 7. WHY THIS HAPPENS .................. Spreadsheet vs. Relational thinking
 * ─────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * SAMPLE TABLE — raw_learning_events (The Anti-Pattern)
 * ─────────────────────────────────────────────────────────────────────
 * ┌────────┬─────────────────────┬──────────────┬────────────┬──────────────────────┬──────────────────┬──────────────────┬──────────────┬──────────┬───────────────┬────────────────┬─────────────────────┬─────────────────────┐
 * │ row_id │ student_email       │ student_name │ course_code│ course_title         │ instructor_email │ instructor_name  │ listed_price │ currency │ payment_id    │ payment_status │ paid_at             │ enrolled_at         │
 * ├────────┼─────────────────────┼──────────────┼────────────┼──────────────────────┼──────────────────┼──────────────────┼──────────────┼──────────┼───────────────┼────────────────┼─────────────────────┼─────────────────────┤
 * │ 1      │ asha@gmail.com      │ Asha         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1001      │ PAID           │ 2026-01-05 10:10:00 │ 2026-01-05 10:12:00 │
 * │ 2      │ asha@gmail.com      │ Asha Sharma  │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1002      │ REFUNDED       │ 2026-01-06 09:00:00 │ 2026-01-06 09:02:00 │
 * │ 3      │ ravi@gmail.com      │ Ravi         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 599.00       │ INR      │ pay_1003      │ PAID           │ 2026-01-10 18:00:00 │ 2026-01-10 18:02:00 │
 * │ 4      │ neha@gmail.com      │ Neha         │ DE201      │ Data Eng Found       │ sara@tuf.com     │ Sara             │ 19.00        │ USD      │ pay_1004      │ PAID           │ 2026-01-12 20:00:00 │ 2026-01-12 20:05:00 │
 * │ 5      │ neha@gmail.com      │ Neha         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1005      │ PENDING        │ NULL                │ 2026-01-15 08:00:00 │
 * │ 6      │ dev@gmail.com       │ Dev          │ LLD301     │ LLD Bootcamp         │ mike@tuf.com     │ Mike             │ 49.00        │ USD      │ pay_1006      │ PAID           │ 2026-01-20 11:30:00 │ 2026-01-20 11:31:00 │
 * │ 7      │ dev@gmail.com       │ Dev          │ LLD301     │ LLD Bootcamp         │ mike@tuf.com     │ Mike             │ 49.00        │ USD      │ pay_1007      │ PAID           │ 2026-01-21 12:00:00 │ 2026-01-21 12:01:00 │
 * │ 8      │ utkarsh@gmail.com   │ Utkarsh      │ DSA401     │ DSA Mastery          │ raj@tuf.com      │ Raj              │ 999.00       │ INR      │ pay_1008      │ PAID           │ 2026-01-22 09:00:00 │ 2026-01-22 09:05:00 │
 * │ 9      │ anuj@gmail.com      │ Anuj         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1009      │ PAID           │ 2026-01-25 10:00:00 │ 2026-01-25 10:02:00 │
 * │ 10     │ ronit@gmail.com     │ Ronit        │ DE201      │ Data Eng Found       │ sara@tuf.com     │ Sara             │ 19.00        │ USD      │ pay_1010      │ PAID           │ 2026-01-25 15:00:00 │ 2026-01-25 15:05:00 │
 * │ 11     │ shivi@gmail.com     │ Shivi        │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1011      │ PAID           │ 2026-01-28 19:00:00 │ 2026-01-28 19:02:00 │
 * │ 12     │ siddhant@gmail.com  │ Siddhant     │ DE201      │ Data Eng Found       │ sara@tuf.com     │ Sara             │ 19.00        │ USD      │ pay_1012      │ REFUNDED       │ 2026-01-29 10:00:00 │ 2026-01-29 10:01:00 │
 * │ 13     │ vinayak@gmail.com   │ Vinayak      │ DSA401     │ DSA Mastery          │ raj@tuf.com      │ Raj              │ 999.00       │ INR      │ pay_1013      │ PAID           │ 2026-02-01 09:00:00 │ 2026-02-01 09:03:00 │
 * │ 14     │ mridul@gmail.com    │ Mridul       │ LLD301     │ LLD Bootcamp         │ mike@tuf.com     │ Mike             │ 49.00        │ USD      │ pay_1014      │ PAID           │ 2026-02-02 14:00:00 │ 2026-02-02 14:02:00 │
 * │ 15     │ prashant@gmail.com  │ Prashant     │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1015      │ PAID           │ 2026-02-03 16:00:00 │ 2026-02-03 16:01:00 │
 * └────────┴─────────────────────┴──────────────┴────────────┴──────────────────────┴──────────────────┴──────────────────┴──────────────┴──────────┴───────────────┴────────────────┴─────────────────────┴─────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 1. THE COMFORT OF FLAT TABLES — Spreadsheet thinking in databases
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ IN THE EARLY DAYS, DATA FEELS HARMLESS.                            │
 * │                                                                    │
 * │ You export everything from a spreadsheet.                          │
 * │ You dump it into a single table.                                   │
 * │ You see rows. You see values.                                      │
 * │ It looks like everything is working.                               │
 * │                                                                    │
 * │ BUT WHAT YOU ARE ACTUALLY STORING:                                 │
 * │   → Not structure, but REPETITION                                  │
 * │   → Not relationships, but COPIES                                  │
 * │   → Not clarity, but the seeds of FUTURE CONFUSION                 │
 * │                                                                    │
 * │ THE FLAT TABLE ILLUSION:                                           │
 * │                                                                    │
 * │   At first glance, raw_learning_events looks COMPLETE:             │
 * │     → Student information                                          │
 * │     → Course information                                           │
 * │     → Instructor details                                           │
 * │     → Payment status                                               │
 * │     → Enrollment timestamps                                        │
 * │                                                                    │
 * │   Everything in one place.                                         │
 * │   That is exactly the PROBLEM.                                     │
 * │                                                                    │
 * │ SQL IS NOT ABOUT STORING EVERYTHING.                               │
 * │ SQL IS ABOUT STRUCTURING RELATIONSHIPS CORRECTLY.                  │
 * │                                                                    │
 * │ This table violates that principle.                                │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 2. THE SAMPLE TABLE — raw_learning_events
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ WHAT THIS TABLE REPRESENTS:                                        │
 * │   Each row = "A learning event"                                    │
 * │   Student buys a course → one row                                  │
 * │   Another student buys → another row                               │
 * │                                                                    │
 * │ WHAT IT CONTAINS (All in one place):                               │
 * │                                                                    │
 * │   ┌─────────────────────────────────────────────────────────┐      │
 * │   │ Entity Type      │ Columns stored                       │      │
 * │   ├──────────────────┼──────────────────────────────────────┤      │
 * │   │ Student          │ student_email, student_name          │      │
 * │   │ Course           │ course_code, course_title            │      │
 * │   │ Instructor       │ instructor_email, instructor_name    │      │
 * │   │ Price            │ listed_price, currency               │      │
 * │   │ Payment          │ payment_id, payment_status, paid_at  │      │
 * │   │ Enrollment       │ enrolled_at                          │      │
 * │   └─────────────────────────────────────────────────────────┘      │
 * │                                                                    │
 * │ This looks convenient. But it's a DATA NIGHTMARE waiting to happen.│
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 3. THE REAL PROBLEMS — Inconsistency, duplication, confusion
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ PROBLEM 1: STUDENT DETAILS REPEATED EVERYWHERE                     │
 * │                                                                    │
 * │   Look at asha@gmail.com:                                          │
 * │     Row 1: 'asha@gmail.com', 'Asha'                                │
 * │     Row 2: 'asha@gmail.com', 'Asha Sharma'                         │
 * │                                                                    │
 * │   Same email. Two different names.                                 │
 * │   Which one is correct?                                            │
 * │                                                                    │
 * │   In a spreadsheet, you might ignore it.                           │
 * │   In a real product:                                               │
 * │     → Support tickets about wrong names                            │
 * │     → Reporting inconsistencies                                    │
 * │     → Trust issues with data                                       │
 * │     → Manual cleanup nightmares                                    │
 * │                                                                    │
 * │ PROBLEM 2: COURSE DETAILS REPEATED                                 │
 * │                                                                    │
 * │   Every time someone buys SQL101, this repeats:                    │
 * │     'SQL101', 'SQL Basics', 'raj@tuf.com', 'Raj'                   │
 * │                                                                    │
 * │   Now imagine:                                                     │
 * │     → 50,000 purchases of SQL101                                   │
 * │     → Course title changes to "SQL Fundamentals"                   │
 * │     → Instructor email changes                                     │
 * │     → Price changes                                                │
 * │                                                                    │
 * │   With this design:                                                │
 * │     → UPDATE 50,000 rows!                                          │
 * │     → Risk of missing some rows                                    │
 * │     → Inconsistent data (some rows updated, some not)              │
 * │     → Slow, painful maintenance                                    │
 * │                                                                    │
 * │ PROBLEM 3: DUPLICATE ENROLLMENT CONFUSION                          │
 * │                                                                    │
 * │   Look at dev@gmail.com, LLD301:                                   │
 * │     Row 6: pay_1006, PAID                                          │
 * │     Row 7: pay_1007, PAID                                          │
 * │                                                                    │
 * │   Same student. Same course. Two successful payments.              │
 * │   Is this:                                                         │
 * │     → A duplicate enrollment?                                      │
 * │     → A payment retry?                                             │
 * │     → A system bug?                                                │
 * │                                                                    │
 * │   The flat table CANNOT ENFORCE business rules like:               │
 * │     → A student cannot enroll in the same course twice             │
 * │     → One enrollment per payment                                   │
 * │                                                                    │
 * │   Data integrity is not enforced.                                  │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 4. STUDENT DETAILS REPEATED — Name inconsistency, update nightmare
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE ASHA PROBLEM:                                                  │
 * │                                                                    │
 * │   SELECT student_email, student_name, COUNT(*)                     │
 * │   FROM raw_learning_events                                         │
 * │   WHERE student_email = 'asha@gmail.com'                           │
 * │   GROUP BY student_email, student_name;                            │
 * │                                                                    │
 * │   Output:                                                          │
 * │   +-----------------+--------------+-------+                       │
 * │   | student_email   | student_name | COUNT |                       │
 * │   +-----------------+--------------+-------+                       │
 * │   | asha@gmail.com  | Asha         | 1     |                       │
 * │   | asha@gmail.com  | Asha Sharma  | 1     |                       │
 * │   +-----------------+--------------+-------+                       │
 * │                                                                    │
 * │   One user, two names. Which is correct?                           │
 * │                                                                    │
 * │   CONSEQUENCES:                                                    │
 * │                                                                    │
 * │   1. Customer Support Confusion:                                   │
 * │      → "Hi, my name is Asha. Why does your system call me          │
 * │        Asha Sharma sometimes?"                                     │
 * │                                                                    │
 * │   2. Reporting Errors:                                             │
 * │      → "How many unique students bought SQL101?"                   │
 * │      → If you count by student_name, Asha appears twice!           │
 * │      → If you count by email, Asha appears once.                   │
 * │      → Which report is correct?                                    │
 * │                                                                    │
 * │   3. Update Nightmare:                                             │
 * │      → To fix Asha's name, you need to update ALL rows             │
 * │        WHERE student_email = 'asha@gmail.com'                      │
 * │      → If you miss one row, inconsistency persists                 │
 * │                                                                    │
 * │   4. No Single Source of Truth:                                    │
 * │      → Student information is NOT stored in one place              │
 * │      → It's scattered across every enrollment row                  │
 * │      → Any change requires updating many rows                      │
 * │                                                                    │
 * │   THE RELATIONAL SOLUTION:                                         │
 * │                                                                    │
 * │   Separate students table:                                         │
 * │     students(id, email, name) → one row per student                │
 * │   Enrollments reference student_id, not email/name                 │
 * │                                                                    │
 * │   Now name changes update ONE row.                                 │
 * │   No inconsistency. No update nightmare.                           │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 5. COURSE DETAILS REPEATED — Title changes affect thousands of rows
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE SQL101 PROBLEM:                                                │
 * │                                                                    │
 * │   SELECT course_code, COUNT(*)                                     │
 * │   FROM raw_learning_events                                         │
 * │   WHERE course_code = 'SQL101';                                    │
 * │                                                                    │
 * │   Let's count how many times SQL101 appears:                       │
 * │   → Rows 1, 2, 3, 5, 9, 11, 15 = 7 rows                            │
 * │                                                                    │
 * │   Now imagine:                                                     │
 * │     → Product team decides: "SQL Basics" should become             │
 * │       "SQL Fundamentals"                                           │
 * │     → Course price changes from 499 to 599                         │
 * │                                                                    │
 * │   WITH FLAT TABLE:                                                 │
 * │                                                                    │
 * │     UPDATE raw_learning_events                                     │
 * │     SET course_title = 'SQL Fundamentals',                         │
 * │         listed_price = 599.00                                      │
 * │     WHERE course_code = 'SQL101';                                  │
 * │                                                                    │
 * │     → Updates 7 rows (now).                                        │
 * │     → With 50,000 enrollments, updates 50,000 rows!                │
 * │     → Takes minutes or hours                                       │
 * │     → Locks table while updating                                   │
 * │     → Risk of partial updates                                      │
 * │                                                                    │
 * │   THE RELATIONAL SOLUTION:                                         │
 * │                                                                    │
 * │   Separate courses table:                                          │
 * │     courses(id, code, title, price, instructor_id)                 │
 * │     → One row per course                                           │
 * │                                                                    │
 * │   Enrollments reference course_id, not copy all course data.       │
 * │                                                                    │
 * │   Now title/price change:                                          │
 * │     UPDATE courses SET title = 'SQL Fundamentals'                  │
 * │     WHERE code = 'SQL101';                                         │
 * │                                                                    │
 * │     → Updates 1 row!                                               │
 * │     → Instant                                                      │
 * │     → All enrollments automatically see the new title              │
 * │     → No inconsistency                                             │
 * │                                                                    │
 * │   This is the power of normalization.                              │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 6. DUPLICATE ENROLLMENT CONFUSION — No business rule enforcement
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
 * │ THE DEV PROBLEM:                                                                                                 │
 * │                                                                                                                  │
 * │   SELECT * FROM raw_learning_events                                                                              │
 * │   WHERE student_email = 'dev@gmail.com'                                                                          │
 * │     AND course_code = 'LLD301';                                                                                  │
 * │                                                                                                                  │
 * │   Output:                                                                                                        │
 * │   +--------+-----------------+--------------+-------------+------------+----------------+---------------------+  |
 * │   | row_id | student_email   | student_name | course_code | payment_id | payment_status | paid_at             |  |
 * │   +--------+-----------------+--------------+-------------+------------+----------------+---------------------+  |
 * │   | 6      | dev@gmail.com   | Dev          | LLD301      | pay_1006   | PAID           | 2026-01-20 11:30:00 |  |
 * │   | 7      | dev@gmail.com   | Dev          | LLD301      | pay_1007   | PAID           | 2026-01-21 12:00:00 |  |
 * │   +--------+-----------------+--------------+-------------+------------+----------------+---------------------+  |
 * │                                                                                                                  │
 * │   QUESTIONS THIS RAISES:                                                                                         │
 * │                                                                                                                  │
 * │   1. Is this a duplicate enrollment?                                                                             │
 * │      → Did Dev enroll twice by mistake?                                                                          │
 * │                                                                                                                  │
 * │   2. Is this a payment retry?                                                                                    │
 * │      → Did first payment fail and second succeed?                                                                │
 * │      → But both show PAID                                                                                        │
 * │                                                                                                                  │
 * │   3. Is this a system bug?                                                                                       │
 * │      → Did the enrollment process allow double enrollment?                                                       │
 * │                                                                                                                  │
 * │   THE FLAT TABLE CANNOT ENFORCE:                                                                                 │
 * │     → UNIQUE constraint on (student_email, course_code)                                                          │
 * │     → Business rule: "One active enrollment per student per course"                                              │
 * │     → Payment validation                                                                                         │
 * │                                                                                                                  │
 * │   CONSEQUENCES:                                                                                                  │
 * │     → Data integrity issues                                                                                      │
 * │     → Wrong reports ("How many students took LLD301?" counts Dev twice)                                          │
 * │     → Customer confusion (Dev might be charged twice)                                                            │
 * │     → Support tickets                                                                                            │
 * │                                                                                                                  │
 * │   THE RELATIONAL SOLUTION:                                                                                       │
 * │                                                                                                                  │
 * │   Separate enrollments table with constraints:                                                                   │
 * │     → UNIQUE(student_id, course_id)                                                                              │
 * │     → Prevents duplicate enrollments                                                                             │
 * │     → Database enforces business rule                                                                            │
 * │                                                                                                                  │
 * │   Now the database itself prevents duplicate enrollments.                                                        │
 * │   No application logic needed.                                                                                   │
 * └──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 7. WHY THIS HAPPENS — Spreadsheet vs. Relational thinking
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE ROOT CAUSE:                                                    │
 * │                                                                    │
 * │   This table is NOT a database design.                             │
 * │   It is a SPREADSHEET DUMP.                                        │
 * │                                                                    │
 * │ SPREADSHEET THINKING:                                              │
 * │                                                                    │
 * │   ┌─────────────────────────────────────────────────────────┐      │
 * │   │ • Store events row by row                               │      │
 * │   │ • Repeat information freely                             │      │
 * │   │ • One big table for everything                          │      │
 * │   │ • Easy to dump and query initially                      │      │
 * │   │ • Feels productive at first                             │      │
 * │   │ • No thought about relationships                        │      │
 * │   └─────────────────────────────────────────────────────────┘      │
 * │                                                                    │
 * │ RELATIONAL THINKING:                                               │
 * │                                                                    │
 * │   ┌─────────────────────────────────────────────────────────┐      │
 * │   │ • Model ENTITIES (students, courses, instructors)       │      │
 * │   │ • Model RELATIONSHIPS (enrollments, payments)           │      │
 * │   │ • Store each entity ONCE                                │      │
 * │   │ • Reference via foreign keys                            │      │
 * │   │ • Database enforces integrity                           │      │
 * │   │ • Updates affect ONE row                                │      │
 * │   │ • Designed for consistency, not convenience             │      │
 * │   └─────────────────────────────────────────────────────────┘      │
 * │                                                                    │
 * │ THE SHIFT IN MINDSET:                                              │
 * │                                                                    │
 * │   From: "How do I store this event?"                               │
 * │   To:   "What entities exist? How are they related?"               │
 * │                                                                    │
 * │   From: "Everything in one table"                                  │
 * │   To:   "Multiple tables connected by relationships"               │
 * │                                                                    │
 * │   From: "Data entry is easy"                                       │
 * │   To:   "Data integrity is guaranteed"                             │
 * │                                                                    │
 * │   From: "I can see everything at once"                             │
 * │   To:   "I can query relationships efficiently"                    │
 * │                                                                    │
 * │ This is why normalization exists.                                  │
 * │ This is why relational databases were invented.                    │
 * │ This is the difference between a spreadsheet and a database.       │
 * └────────────────────────────────────────────────────────────────────┘
 */
